In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import importlib

import model_utils
importlib.reload(model_utils)
from model_utils import *


In [ ]:
class SkipTFUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Intercept Keras/TensorFlow deserialization functions & model objects
        if "keras" in module or "tensorflow" in module or name in ["Functional", "Sequential", "Model"]:
            # Return a dummy placeholder class instead of recreating the Keras object
            return DummyKerasModel
        return super().find_class(module, name)

class DummyKerasModel:
    """Placeholder to safely swallow Keras unpickling calls without crashing."""
    def __init__(self, *args, **kwargs):
        pass
    
    @classmethod
    def _unpickle_model(cls, *args, **kwargs):
        return "<Skipped Keras Model>"

Models Results

In [ ]:
n_clients = [10, 15, 20]
iids = ['IID', 'Non-IID']
seeds = [42, 123, 456]
seed = seeds[0]

all_experiments = {}

for iid in iids:
    for clients in n_clients:
        experiment = f'experiments/TON-n_{clients}-{iid}-seed_{seed}.pkl'
        with open(experiment, 'rb') as f:
            results = SkipTFUnpickler(f).load()

            all_experiments[(iid, clients)] = results["metrics"]

In [ ]:
def generate_latex_table(exp_dict):
    """
    exp_dict expected structure:
    {
        ('IID', 10): results_iid_10['metrics'],
        ('IID', 15): results_iid_15['metrics'],
        ('IID', 20): results_iid_20['metrics'],
        ('Non-IID', 10): results_non_iid_10['metrics'],
        ('Non-IID', 15): results_non_iid_15['metrics'],
        ('Non-IID', 20): results_non_iid_20['metrics'],
    }
    """
    '''metrics_to_show = [
        ('Global_MSE', 'Global MSE', '{:.6f}'),
        ('Global_MAE', 'Global MAE', '{:.6f}'),
        ('Client_Loss_Mean', 'Client Loss Mean', '{:.6f}'),
        ('Client_Loss_Std', 'Client Loss Std', '{:.6f}'),
        ('Silhouette_Score', 'Silhouette Score', '{:.4f}'),
        ('Correlation_Distortion', 'Correlation Distortion', '{:.4f}'),
        ('Compression_Ratio', 'Compression Ratio', '{:.2f}'),
        ('Active_Neuron_Ratio', 'Active Neuron Ratio', '{:.4f}'),
        ('Throughput_ms_per_sample', 'Throughput (ms/sample)', '{:.4f}'),
        ('Payload_MB_per_Client', 'Payload / Client (MB)', '{:.2f}')
    ]'''

    metrics_to_show = [
            ('Global_MSE', 'Global MSE', '{:.2E}'),
            ('Global_MAE', 'Global MAE', '{:.2E}'),
            ('Client_Loss_Mean', 'Client Loss Mean', '{:.2E}'),
            ('Client_Loss_Std', 'Client Loss Std', '{:.2E}'),
            ('Silhouette_Score', 'Silhouette Score', '{:.2E}'),
            ('Correlation_Distortion', 'Correlation Distortion', '{:.2E}'),
            ('Compression_Ratio', 'Compression Ratio', '{:.2E}'),
            ('Active_Neuron_Ratio', 'Active Neuron Ratio', '{:.2E}'),
            ('Throughput_ms_per_sample', 'Throughput (ms/sample)', '{:.2E}'),
            ('Payload_MB_per_Client', 'Payload / Client (MB)', '{:.2E}')
        ]
    
    cols = [('IID', 10), ('IID', 15), ('IID', 20), 
            ('Non-IID', 10), ('Non-IID', 15), ('Non-IID', 20)]

    print("System Performance Across Client Scalability and Data Distributions")
    print("IID Distribution\tNon-IID Distribution")
    print("Metric\t10 Clients\t15 Clients\t20 Clients\t10 Clients\t15 Clients\t20 Clients")

    for key, label, fmt in metrics_to_show:
        row_vals = []
        for c in cols:
            val = exp_dict.get(c, {}).get(key, 0.0)
            row_vals.append(fmt.format(val))
        print(f"{label}\t", "\t".join(row_vals))

# Example invocation:
generate_latex_table(all_experiments)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Set publication style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.dpi': 300})

def plot_bar_graph_experiments(exp_dict, save_path="fl_clients_bargraph.png"):
    """
    exp_dict structure expected:
    {
        ('IID', 10): results_iid_10['metrics'],
        ('IID', 15): results_iid_15['metrics'],
        ('IID', 20): results_iid_20['metrics'],
        ('Non-IID', 10): results_non_iid_10['metrics'],
        ('Non-IID', 15): results_non_iid_15['metrics'],
        ('Non-IID', 20): results_non_iid_20['metrics'],
    }
    """
    client_counts = [10, 15, 20]
    x = np.arange(len(client_counts))  # Label locations
    width = 0.35  # Bar width

    # Extract metrics for IID and Non-IID
    iid_mse = [exp_dict.get(('IID', c), {}).get('Global_MSE', 0.0) for c in client_counts]
    non_iid_mse = [exp_dict.get(('Non-IID', c), {}).get('Global_MSE', 0.0) for c in client_counts]

    iid_std = [exp_dict.get(('IID', c), {}).get('Client_Loss_Std', 0.0) for c in client_counts]
    non_iid_std = [exp_dict.get(('Non-IID', c), {}).get('Client_Loss_Std', 0.0) for c in client_counts]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # =========================================================================
    # BAR GRAPH 1: Global MSE Across Client Counts & Distributions
    # =========================================================================
    rects1 = axes[0].bar(x - width/2, iid_mse, width, label='IID', color='#1f77b4', edgecolor='black', alpha=0.9)
    rects2 = axes[0].bar(x + width/2, non_iid_mse, width, label='Non-IID', color='#d62728', edgecolor='black', alpha=0.9)

    axes[0].set_title("Global Reconstruction MSE", fontweight="bold", pad=12)
    axes[0].set_xlabel("Number of Clients", fontweight="bold")
    axes[0].set_ylabel("Global MSE", fontweight="bold")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"{c} Clients" for c in client_counts])
    axes[0].legend(loc="upper left")
    axes[0].grid(True, linestyle="--", alpha=0.5, axis='y')

    # Add numeric value annotations above bars
    for rect in rects1 + rects2:
        h = rect.get_height()
        if h > 0:
            axes[0].annotate(f"{h:.6f}",
                                xy=(rect.get_x() + rect.get_width() / 2, h),
                                xytext=(0, 3), textcoords="offset points",
                                ha='center', va='bottom', fontsize=9, fontweight='bold')

    # =========================================================================
    # BAR GRAPH 2: Client Loss Std Dev (Heterogeneity Variance)
    # =========================================================================
    rects3 = axes[1].bar(x - width/2, iid_std, width, label='IID', color='#1f77b4', edgecolor='black', alpha=0.9)
    rects4 = axes[1].bar(x + width/2, non_iid_std, width, label='Non-IID', color='#d62728', edgecolor='black', alpha=0.9)

    axes[1].set_title("Client Loss Variance (Non-IID Divergence)", fontweight="bold", pad=12)
    axes[1].set_xlabel("Number of Clients", fontweight="bold")
    axes[1].set_ylabel("Client Loss Std Dev", fontweight="bold")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f"{c} Clients" for c in client_counts])
    axes[1].legend(loc="upper left")
    axes[1].grid(True, linestyle="--", alpha=0.5, axis='y')

    # Add numeric value annotations above bars
    for rect in rects3 + rects4:
        h = rect.get_height()
        if h > 0:
            axes[1].annotate(f"{h:.6f}",
                                xy=(rect.get_x() + rect.get_width() / 2, h),
                                xytext=(0, 3), textcoords="offset points",
                                ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f"Bar graph successfully saved to {save_path}")

# Run function (pass your populated dictionary containing the 6 experiments)
plot_bar_graph_experiments(all_experiments)